# Photonic Detection — GHz/THz Signals, STEAM Camera, Time-Stretch
Jalali lab: serial time-encoded amplified microscopy, dispersive Fourier transform, single-pixel detection at 100 GHz rates.

In [ ]:
from sympy import *
from IPython.display import display, Markdown
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — The Detection Bottleneck: Why GHz Cameras Are Hard

In [ ]:
section('Detection Speed Limits')

display(Markdown(r'''
| Detector | Frame rate | Pixel count | Limitation |
|---|---|---|---|
| CMOS camera | 1 kfps | 10⁶ | readout bandwidth |
| sCMOS | 100 kfps | 10⁴ | charge transfer speed |
| Streak camera | 10 Gfps | 1D only | sweep voltage slew rate |
| **STEAM camera** | **100 Gfps** | **serial 1D** | amplified dispersive FT |
| Single-pixel + DFT | **THz sampling** | 1 pixel | ADC bandwidth |

**Fundamental limit**: a camera with $N$ pixels needs $N \times f_{frame}$ samples/sec from the ADC.
At 1 Mpix × 1 Gfps = 10¹⁵ samples/sec → impossible electronically.

**Jalali solution**: encode the 2D image into a 1D temporal serial stream using *dispersive Fourier transform*,
then digitize with a single fast ADC.
'''))

# Nyquist bandwidth
f_s, B = symbols('f_s B', positive=True)
step('Nyquist: signal bandwidth B requires sampling rate f_s ≥ 2B')
step('For 100 GHz optical signal: f_s ≥', 2*100e9, 'samples/sec = 200 GSa/s')
step('State of art ADC (2025): ~240 GSa/s at 8-bit → barely sufficient')

## §2 — Time-Stretch Dispersive Fourier Transform (TS-DFT)

In [ ]:
section('TS-DFT: Slow Down Light to Digitize It')

omega, t, z, beta2 = symbols('omega t z beta_2', real=True)
D, lam, c = symbols('D lambda c', positive=True)

display(Markdown(r'''
**Principle**: broadband pulse propagates through dispersive fiber.
GVD $\beta_2$ maps frequency → time (group delay):
$$\tau(\omega) = \beta_2 \, z \, \omega$$
The temporal waveform *is* the spectrum — one fast photodetector replaces a spectrometer.
'''))

# Group delay
phi = Rational(1,2)*beta2*omega**2*z
tau = diff(phi, omega)
step('Dispersion phase φ(ω) =', phi)
step('Group delay τ(ω) = dφ/dω =', tau)

# Time stretch factor
display(Markdown(r'''
**Time-stretch factor** $M$: the pulse is stretched in time by factor
$$M = 1 + \frac{|\beta_2^{(2)} z_2|}{|\beta_2^{(1)} z_1|}$$
where fiber 1 = dispersive fiber (TS-DFT), fiber 2 = compensation fiber.
Typical $M = 100\text{–}1000\times$ → 100 GHz signal digitized at 100 MHz ADC.
'''))

M = symbols('M', positive=True)
f_optical, f_adc = symbols('f_optical f_ADC', positive=True)
step('Required ADC rate after stretch:', Eq(f_adc, f_optical / M))
step('Example: 1 THz optical, M=1000 → ADC rate =', '1 GHz — achievable')

## §3 — STEAM Camera: Serial Time-Encoded Amplified Microscopy

In [ ]:
section('STEAM Camera')

display(Markdown(r'''
**STEAM pipeline** (Goda et al., Jalali lab, Nature 2009):

```
Broadband pulse
      │
      ▼
Spatial disperser (diffraction grating or prism)
  → maps wavelength to spatial position
      │
      ▼
Object plane (image encoded in spectrum)
  → each spatial pixel modulates a different wavelength
      │
      ▼
Temporal disperser (dispersive fiber, TS-DFT)
  → maps wavelength to time
      │
      ▼
Raman amplifier (compensates fiber loss)
      │
      ▼
Single photodetector + ADC
  → 1D serial waveform = 1 image row
      │
      ▼
Repeat for each pulse → frame sequence
```

**Result**: 100 billion frames per second, limited only by laser repetition rate and ADC speed.
'''))

# Frame rate
f_rep, N_pix = symbols('f_rep N_pixels', positive=True)
step('Frame rate = laser repetition rate f_rep')
step('Pixel rate = f_rep × N_pixels per pulse')
step('Jalali lab: f_rep = 36.7 MHz, N_pix ~ 3000 → 110 Gpixel/sec')

## §4 — Signal Chain: Photons to Bits

In [ ]:
section('Signal Chain')

display(Markdown(r'''
```
Optical pulse (broadband, 1550 nm center)
  Power: P(t) [mW]  Bandwidth: Δλ ~ 10–100 nm
         │
         ▼
Photodetector (InGaAs, responsivity R [A/W])
  I_ph = R · P(t)
         │
         ▼
Transimpedance amplifier (TIA, gain Z_T [Ω])
  V_out = Z_T · I_ph = Z_T · R · P(t)
         │
         ▼
ADC (f_s samples/sec, N_bits resolution)
  Dynamic range = 6.02·N_bits + 1.76 dB
         │
         ▼
Digital signal processing (GS phase retrieval)
```
'''))

R_resp, P, Z_T, N_bits = symbols('R P Z_T N', positive=True)

I_ph = R_resp * P
V_out = Z_T * I_ph
step('Photocurrent I_ph = R·P =', I_ph)
step('TIA output V = Z_T·R·P =', V_out)

ENOB = N_bits  # effective number of bits
DR_dB = 6.02*ENOB + 1.76
step('ADC dynamic range (dB) =', DR_dB)
step('At 8 bits:', DR_dB.subs(N_bits, 8), 'dB')
step('At 12 bits:', DR_dB.subs(N_bits, 12), 'dB')

# Shot noise
q, Delta_f = symbols('q Delta_f', positive=True)
I_shot = sqrt(2*q*I_ph*Delta_f)
step('Shot noise current σ_shot = √(2qI·Δf) =', I_shot)
SNR = I_ph / I_shot
step('SNR_shot = I_ph / σ_shot =', simplify(SNR))
step('= √(I_ph / (2q·Δf)) — improves with more photons')

## §5 — THz Detection: Methods and Limits

In [ ]:
section('THz Detection')

display(Markdown(r'''
**THz gap** (0.1–10 THz): too fast for electronics, too slow for optics.
Three detection strategies:

| Method | Principle | Bandwidth | Notes |
|---|---|---|---|
| Electro-optic sampling | THz field modulates optical probe via Pockels effect | DC–100 THz | coherent, phase-sensitive |
| Photoconductive antenna | THz pulse gates GaAs switch | DC–5 THz | room temp |
| Bolometer | thermal power detection | broadband | needs 4 K cooling |
| **TS-DFT + single pixel** | optical encoding → serial readout | **up to 10 THz** | room temp, coherent |

**Electro-optic sampling** — the phase-sensitive coherent method:
'''))

# EO sampling
E_THz, r41, n, L_EO, lam_probe = symbols('E_THz r_41 n L lambda_probe', positive=True)
Delta_phi = pi * n**3 * r41 * E_THz * L_EO / lam_probe
step('EO phase shift Δφ =', Delta_phi)
step('For ZnTe: r₄₁=4 pm/V, n=2.85, L=1 mm, λ=800 nm')
vals = [(n, 2.85), (r41, 4e-12), (L_EO, 1e-3), (lam_probe, 800e-9)]
Dphi_numeric = Delta_phi.subs(vals)
step('Δφ/E_THz =', float(Dphi_numeric.subs(E_THz, 1)), 'rad per V/m')

display(Markdown(r'''
**Key**: EO sampling measures $E_{THz}(t)$ directly (field, not intensity) →
full complex signal → **no phase retrieval needed** for THz time-domain spectroscopy.

For optical intensity signals (STEAM), phase retrieval (GS/TDGSA) is essential.
'''))

## §6 — Rogue Wave Detection: The RogueGuard Problem

In [ ]:
section('Optical Rogue Waves')

display(Markdown(r'''
**Optical rogue waves**: rare, extremely intense pulses in a fiber laser or
supercontinuum source. Analogous to ocean rogue waves — amplitude far exceeds
significant wave height.

**Statistics**: intensity PDF has a heavy tail (L-shaped) instead of Gaussian.
$$P(I > I_{thresh}) \sim e^{-I/I_0} \cdot I^{-\alpha}, \quad \alpha > 1$$

**Why they matter for STEAM / TDGSA**:
- Rogue pulse saturates ADC → corrupts frame
- GS algorithm diverges when seed intensity has outliers
- Must detect and gate out rogue events in real time

**RogueGuard** (1U rack unit):
- RPi CM4 + dual ADC (2 channels, 1 GSa/s each)
- TD-GS phase retrieval on each pulse
- CNN classifier: rogue vs. normal
- Hardware trigger to gate downstream detector
'''))

# Rogue wave threshold: significant wave height
I_mean, I_thresh = symbols('I_mean I_thresh', positive=True)
# Ocean definition: rogue if H > 2 * H_s (significant wave height)
# Optical analog: rogue if I > 2 * <I>
step('Rogue threshold (optical): I_rogue > 2 · ⟨I⟩')
rogue_prob_gaussian = erfc(sqrt(2))  # P(I > 2σ) for Gaussian
step('Gaussian tail P(I > 2⟨I⟩) =', float(rogue_prob_gaussian.evalf()), '(~2%)')
step('Measured optical rogue rate: 10⁻⁴ to 10⁻⁶ → heavy tail confirmed')

## §7 — ADC Specs: What the Hardware Actually Sees

In [ ]:
section('ADC and Signal Specs')

display(Markdown(r'''
| Parameter | Symbol | Typical value | Units |
|---|---|---|---|
| Laser rep rate | $f_{rep}$ | 36.7–100 | MHz |
| Pulse duration (before stretch) | $T_0$ | 0.1–10 | ps |
| Stretch factor | $M$ | 100–10000 | — |
| Pulse duration (after stretch) | $M T_0$ | 10–100 | ns |
| ADC sample rate | $f_s$ | 1–240 | GSa/s |
| ADC bits | $N$ | 8–12 | bits |
| Fiber GVD | $\beta_2$ | −20 | ps²/km (SMF-28) |
| Dispersion $D$ | $D$ | 17 | ps/(nm·km) |
| Required $|\beta_2 z|$ for TDGSA | — | ≥5000 | ps² |
| Fiber length for TDGSA | $z$ | ≥250 | km (or DCF 25 km) |
'''))

# Relationship between D and beta2
lam, c = symbols('lambda c', positive=True)
beta2_from_D = symbols('D_disp') * lam**2 / (2*pi*c)
step('β₂ = −D·λ²/(2πc)  [D in ps/(nm·km), λ in nm]')

# Numeric for SMF-28 at 1550 nm
D_num = 17e-3  # ps/(nm·m) = ps²/m·nm... let's use consistent units
lam_num = 1550  # nm
c_num = 3e8 * 1e-9 * 1e12  # nm/ps
beta2_num = D_num * lam_num**2 / (2*np.pi*c_num)  # ps²/m
step(f'SMF-28: β₂ ≈ {beta2_num*1000:.2f} ps²/km at 1550 nm')
step(f'For |β₂z| = 5000 ps²: z = {5000/(abs(beta2_num)*1000):.0f} km of SMF-28')
step('→ Use DCF (D = −100 ps/nm·km): only ~40 km needed')

## §8 — Putting It Together: TDGSA Signal Flow at 100 GHz

In [ ]:
section('TDGSA at 100 GHz')

display(Markdown(r'''
```
Mode-locked laser (f_rep = 100 MHz, T_0 = 1 ps, bandwidth 10 nm @ 1550 nm)
         │
         ├──────────────────────────────────┐
         │ (reference arm)                  │ (dispersive arm)
         ▼                                  ▼
   Photodetector                   DCF fiber (~40 km, β₂z = −5000 ps²)
   I_t(t) = |ψ(t)|²                         │
         │                          Photodetector
         │                          I_d(t) = |ψ_d(t)|²
         └──────────┬───────────────┘
                    ▼
              Dual ADC (1 GSa/s each, 8-bit)
                    │
                    ▼
              TDGSA (GS with dispersion constraint)
              50 iterations, |β₂z| = 5000 ps²
                    │
                    ▼
              Recovered phase φ(ω)
              Complex field ψ(t) = |ψ|·e^(iφ)
                    │
                    ▼
              Edge map, rogue wave detection, classification
```

**Throughput**: 100 MHz × 1000 samples/pulse = 100 GSa/s equivalent  
**Phase accuracy**: improves with |β₂z|, degrades with ADC noise  
**Key requirement**: measurement diversity (correlation < 0.95 between I_t and I_d)
'''))

# Convergence metric
n_iter = symbols('k', integer=True, nonnegative=True)
epsilon = symbols('epsilon', positive=True)
step('GS error metric at iteration k:')
display(Math(r'\epsilon^{(k)} = \frac{\||\psi^{(k)}|^2 - I_t\|_2}{\|I_t\|_2}'))
step('Convergence criterion: ε < 10⁻³ or k = 50 iterations')
step('With |β₂z| = 5000 ps²: typical convergence at k = 20–30')